In [11]:
# Step 1 – Install Academic Research Tools
%pip install -q \
  llama-index \
  llama-index-llms-replicate \
  llama-index-embeddings-huggingface \
  llama-index-readers-file \
  llama-index-packs-fusion-retriever \
  sentence-transformers \
  huggingface_hub[hf_xet] \
  hf_xet \
  certifi \
  python-certifi-win32 \
  truststore \
  nest-asyncio \
  requests \
  replicate \
  pytesseract \
  pdf2image \
  Pillow \
  PyMuPDF

import nest_asyncio
nest_asyncio.apply()
print("✅ Installation complete (including OCR deps).")

# NOTE: System dependencies required for OCR:
# - Tesseract OCR executable (installable from https://github.com/tesseract-ocr/tesseract)
# - Poppler utils (for pdf2image) available via package managers or https://poppler.freedesktop.org/
# If those executables are not installed, the OCR fallback will print instructions.

Note: you may need to restart the kernel to use updated packages.
✅ Installation complete (including OCR deps).


In [13]:
# Console / Logger helper (widget-free for VS Code compatibility)
from datetime import datetime
import logging

def console_log(msg, level='INFO'):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{ts}] {level}: {msg}")

class ConsoleHandler(logging.Handler):
    def emit(self, record):
        console_log(self.format(record), record.levelname)

root_logger = logging.getLogger()
if not any(isinstance(h, ConsoleHandler) for h in root_logger.handlers):
    handler = ConsoleHandler()
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

root_logger.setLevel(logging.INFO)
console_log("Console initialized — logs will appear in cell output.", "OK")

[2026-03-07 08:49:21] OK: Console initialized — logs will appear in cell output.


In [14]:
# Diagnostics: verify Tesseract and Poppler (pdf2image) availability
import shutil
import os

console_log("Running diagnostics: checking system OCR dependencies...", "INFO")

# Tesseract check
tess_path = shutil.which("tesseract")
if tess_path:
    try:
        import pytesseract
        v = pytesseract.get_tesseract_version()
        console_log(f"Tesseract found: {tess_path} — version {v}", "OK")
    except Exception as e:
        console_log(f"Tesseract found at {tess_path} but pytesseract error: {e}", "WARN")
else:
    console_log("Tesseract executable not found in PATH.", "ERROR")
    console_log("Install Tesseract: https://github.com/tesseract-ocr/tesseract or `choco install tesseract`", "INFO")

# Poppler check (pdftoppm or pdftocairo)
poppler_bin = shutil.which("pdftoppm") or shutil.which("pdftocairo")
if poppler_bin:
    console_log(f"Poppler utility found: {poppler_bin}", "OK")
else:
    console_log("Poppler utilities (pdftoppm/pdftocairo) not found in PATH.", "ERROR")
    console_log("Install Poppler: https://poppler.freedesktop.org/ or `choco install poppler`", "INFO")

# Optional pdf2image test if a sample PDF exists
sample_pdf = os.path.join("academic_data", "source_material.pdf")
if os.path.exists(sample_pdf):
    if poppler_bin:
        try:
            from pdf2image import convert_from_path
            imgs = convert_from_path(sample_pdf, dpi=50, first_page=1, last_page=1)
            console_log("pdf2image conversion test succeeded (Poppler working).", "OK")
        except Exception as e:
            console_log(f"pdf2image conversion test failed: {e}", "ERROR")
    else:
        console_log("Skipping pdf2image test because Poppler not found.", "WARN")
else:
    console_log(f"No sample PDF at {sample_pdf}; skipping conversion test.", "INFO")

console_log("Diagnostics complete.", "OK")


[2026-03-07 08:49:31] INFO: Running diagnostics: checking system OCR dependencies...
[2026-03-07 08:49:31] ERROR: Tesseract executable not found in PATH.
[2026-03-07 08:49:31] INFO: Install Tesseract: https://github.com/tesseract-ocr/tesseract or `choco install tesseract`
[2026-03-07 08:49:31] ERROR: Poppler utilities (pdftoppm/pdftocairo) not found in PATH.
[2026-03-07 08:49:31] INFO: Install Poppler: https://poppler.freedesktop.org/ or `choco install poppler`
[2026-03-07 08:49:31] WARN: Skipping pdf2image test because Poppler not found.
[2026-03-07 08:49:31] OK: Diagnostics complete.


In [16]:
# Step 2: Configure IBM Granite & Security Guardrails
import os
from getpass import getpass
import certifi
from llama_index.core import Settings
from llama_index.llms.replicate import Replicate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Networking hardening for enterprise SSL + HF transport
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
os.environ.setdefault("CURL_CA_BUNDLE", certifi.where())

# On Windows/corporate networks, this merges Windows cert store into certifi trust
try:
    import certifi_win32  # noqa: F401
    print("✅ Windows certificate store bridge enabled (certifi-win32).")
except Exception as e:
    print(f"⚠️ certifi-win32 not active ({e}); using certifi defaults.")

# Enter your REPLICATE_API_KEY safely (env var first, then prompt)
replicate_token = os.getenv("REPLICATE_API_TOKEN", "").strip()
if not replicate_token:
    try:
        replicate_token = getpass("Enter REPLICATE_API_TOKEN: ").strip()
    except Exception:
        replicate_token = input("Enter REPLICATE_API_TOKEN: ").strip()

if not replicate_token:
    raise ValueError("REPLICATE_API_TOKEN is required to continue.")

os.environ["REPLICATE_API_TOKEN"] = replicate_token

# ACADEMIC SHIELD FIX: Granite 3.1 with Extended Patience
llm = Replicate(
    model="ibm-granite/granite-3.1-8b-instruct",
    temperature=0.1, 
    context_window=128000,
    is_chat_model=True,
    request_timeout=600.0, # Increased to 10 minutes for complex Statistics/Leadership PDFs
    system_prompt=(
        "You are an academic research assistant. Answer ONLY using the provided context. "
        "If the answer is not in the text, say 'Information not found in source.' "
        "Provide facts in 2-4 evidence bullets with absolute objectivity."
    )
)

# Embedding model with robust fallback path
try:
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    print("✅ Primary embedding model loaded: BAAI/bge-small-en-v1.5")
except Exception as e1:
    print(f"⚠️ Primary embedding load failed: {e1}")
    try:
        embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
        print("✅ Fallback embedding model loaded: all-MiniLM-L6-v2")
    except Exception as e2:
        from llama_index.core.embeddings import MockEmbedding
        embed_model = MockEmbedding(embed_dim=384)
        print(f"⚠️ HF downloads unavailable; using MockEmbedding fallback ({e2}).")

Settings.llm = llm
Settings.embed_model = embed_model

print("🚀 Granite 3.1 Ready with Academic Shield (Timeout: 300s)")

✅ Windows certificate store bridge enabled (certifi-win32).
[2026-03-07 08:50:11] INFO: 08:50:11 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
[2026-03-07 08:50:11] INFO: 08:50:11 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
[2026-03-07 08:50:11] INFO: 08:50:11 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
[2026-03-07 08:50:11] INFO: 08:50:11 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
[2026-03-07 08:50:11] INFO: 08:50:11 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
[2026-03-07 08:50:22] INFO: 08:50:22 - INFO - 1 prompt is loaded, with the key: query
[2026-03-07 08:50:22] INFO: 08:50:22 - INFO - 1 prompt is loaded, with the key: query
[2026-03-07 08:50:22] INFO: 08:50:22 - INFO - 1 prompt is loaded, with the key: query
[2026-03-07 08:50:22] INFO: 08:50:22 - INFO - 1 prompt is loaded, with the key: query
[2026-03-07 08:50:22] INFO: 08:50:22 - INFO - 1 prompt is loaded, 

In [17]:
# Step 3: Automated PDF Ingestion (Google Drive)
import os
import re
import requests

def extract_drive_file_id(drive_url: str) -> str:
    """Extract Google Drive file id from common share URL formats."""
    patterns = [
        r"/d/([A-Za-z0-9_-]+)",
        r"[?&]id=([A-Za-z0-9_-]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, drive_url)
        if match:
            return match.group(1)
    raise ValueError("Could not extract a Google Drive file id from the provided link.")

def download_pdf_from_drive(drive_url: str, save_path: str, session: requests.Session | None = None) -> None:
    """Download a Drive file and ensure the saved artifact is a valid PDF."""
    session = session or requests.Session()
    file_id = extract_drive_file_id(drive_url)

    base_url = "https://drive.google.com/uc"
    params = {"export": "download", "id": file_id}
    response = session.get(base_url, params=params, stream=True)

    token = None
    for key, val in response.cookies.items():
        if key.startswith("download_warning"):
            token = val
            break

    if not token:
        preview_html = response.content[:200000].decode("utf-8", errors="ignore")
        token_match = re.search(r"confirm=([0-9A-Za-z-_]+)", preview_html)
        if token_match:
            token = token_match.group(1)

    if token:
        params["confirm"] = token
        response = session.get(base_url, params=params, stream=True)

    response.raise_for_status()

    with open(save_path, "wb") as file_obj:
        for chunk in response.iter_content(chunk_size=32768):
            if chunk:
                file_obj.write(chunk)

    with open(save_path, "rb") as file_obj:
        header = file_obj.read(5)

    if header != b"%PDF-":
        with open(save_path, "rb") as file_obj:
            preview = file_obj.read(400).decode("utf-8", errors="ignore")
        os.remove(save_path)
        raise ValueError(
            "Downloaded file is not a valid PDF. "
            "Google Drive likely returned an HTML page (permissions/login/interstitial). "
            "Set sharing to 'Anyone with the link: Viewer' and retry. "
            f"Preview: {preview[:120]!r}"
        )

    console_log(f"Document secured: {save_path}", "OK")

def extract_text_or_ocr(pdf_path: str, dpi: int = 300) -> str:
    """Use PyMuPDF extraction first; fall back to OCR and return chosen source file path."""
    try:
        import fitz  # PyMuPDF
    except Exception:
        fitz = None
        console_log("PyMuPDF not available; OCR fallback may be required.", "WARN")

    extracted_text = ""
    if fitz is not None:
        try:
            with fitz.open(pdf_path) as doc:
                for page in doc:
                    extracted_text += page.get_text() + "\n"
        except Exception as exc:
            console_log(f"PyMuPDF extraction error: {exc}", "WARN")
            extracted_text = ""

    if len(extracted_text.strip()) > 50:
        console_log("PDF contains extractable text (PyMuPDF).", "OK")
        return pdf_path

    console_log("No reliable extractable text found; using OCR fallback.", "WARN")
    try:
        from pdf2image import convert_from_path
        import pytesseract
    except Exception:
        console_log(
            "Missing OCR packages/dependencies (pdf2image, pytesseract, Poppler, Tesseract).",
            "ERROR",
        )
        return pdf_path

    try:
        _ = pytesseract.get_tesseract_version()
    except Exception:
        console_log("Tesseract executable not found. Install it and rerun Step 3.", "ERROR")
        return pdf_path

    try:
        images = convert_from_path(pdf_path, dpi=dpi)
        ocr_text = ""
        for image in images:
            ocr_text += pytesseract.image_to_string(image) + "\n"

        txt_path = pdf_path + ".ocr.txt"
        with open(txt_path, "w", encoding="utf-8") as file_obj:
            file_obj.write(ocr_text)

        console_log(f"OCR complete: {txt_path}", "OK")
        return txt_path
    except Exception as exc:
        console_log(f"OCR failed: {exc}", "ERROR")
        return pdf_path

drive_link = input("📌 Paste Google Drive Link: ").strip()
DATA_DIR = "academic_data"
os.makedirs(DATA_DIR, exist_ok=True)

pdf_path = os.path.join(DATA_DIR, "source_material.pdf")
download_pdf_from_drive(drive_link, pdf_path)

# source_file is consumed by Step 4 and can be either PDF or OCR text file
source_file = extract_text_or_ocr(pdf_path)
console_log(f"Using source file: {source_file}", "OK")

[2026-03-07 08:58:40] OK: Document secured: academic_data\source_material.pdf
[2026-03-07 08:58:40] OK: PDF contains extractable text (PyMuPDF).
[2026-03-07 08:58:40] OK: Using source file: academic_data\source_material.pdf


In [18]:
# Step 4: Semantic Chunking for Contextual Integrity
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SemanticSplitterNodeParser

# 'source_file' is produced by the previous cell and may be a PDF or a .ocr.txt file
documents = SimpleDirectoryReader(input_files=[source_file]).load_data()
parser = SemanticSplitterNodeParser(
    buffer_size=3,
    breakpoint_percentile_threshold=95,
    embed_model=embed_model
)

nodes = parser.get_nodes_from_documents(documents)

# Absolute Referencing Metadata
for n in nodes:
    n.metadata["source"] = os.path.basename(source_file)

console_log(f"Created {len(nodes)} high-quality semantic nodes from {os.path.basename(source_file)}.", "OK")


[2026-03-07 08:59:54] OK: Created 35 high-quality semantic nodes from source_material.pdf.


In [19]:
# Step 5: Advanced Query Fusion (Local Pack, no remote download)
import os
import sys
from pathlib import Path
import importlib.util

local_pack_root = Path("query_rewriting_pack").resolve()
if str(local_pack_root) not in sys.path:
    sys.path.insert(0, str(local_pack_root))

try:
    from llama_index.packs.fusion_retriever.query_rewrite.base import QueryRewritingRetrieverPack
except Exception as import_exc:
    # Fallback: load class directly from file path
    base_file = local_pack_root / "llama_index" / "packs" / "fusion_retriever" / "query_rewrite" / "base.py"
    if not base_file.exists():
        raise FileNotFoundError(
            f"Local pack file not found: {base_file}. Ensure query_rewriting_pack exists in the workspace."
        ) from import_exc
    spec = importlib.util.spec_from_file_location("local_query_rewrite_base", str(base_file))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    QueryRewritingRetrieverPack = module.QueryRewritingRetrieverPack

query_rewriting_pack = QueryRewritingRetrieverPack(
    nodes,
    chunk_size=256,
    vector_similarity_top_k=5,
    num_queries=2 # Changed from 6 to 2 to prevent "Read Operation Timeout"
)
print("🚀 Search Engine optimized for stability.")
console_log("Advanced Query Fusion Engine Ready (local pack)", "OK")

🚀 Search Engine optimized for stability.
[2026-03-07 09:00:25] OK: Advanced Query Fusion Engine Ready (local pack)


In [20]:
# Step 6: The Research Loop (Q&A)
def run_academic_query(question):
    try:
        response = query_rewriting_pack.run(question)
        return response
    except Exception as e:
        console_log(f"Query error: {e}", "ERROR")
        return f"⚠️ Error: {e}"

console_log("--- ACADEMIC RESEARCH MODE ---", "INFO")
while True:
    query = input("\n🟦 Research Question: ").strip()
    if query.lower() == "end": break

    answer = run_academic_query(query)
    console_log(f"FACTUAL ANALYSIS (Granite 3.0): {answer}", "INFO")
    print(f"\n🧠 FACTUAL ANALYSIS (Granite 3.0):\n{answer}")
    console_log("Reference: Absolute Reference to source_material.pdf", "INFO")
    print("\n📍 REFERENCE: Absolute Reference to source_material.pdf")

[2026-03-07 09:00:31] INFO: --- ACADEMIC RESEARCH MODE ---


[2026-03-07 09:00:46] INFO: 09:00:46 - INFO - HTTP Request: POST https://api.replicate.com/v1/models/ibm-granite/granite-3.1-8b-instruct/predictions "HTTP/1.1 201 Created"
[2026-03-07 09:00:46] INFO: 09:00:46 - INFO - HTTP Request: POST https://api.replicate.com/v1/models/ibm-granite/granite-3.1-8b-instruct/predictions "HTTP/1.1 201 Created"
[2026-03-07 09:00:46] INFO: 09:00:46 - INFO - HTTP Request: POST https://api.replicate.com/v1/models/ibm-granite/granite-3.1-8b-instruct/predictions "HTTP/1.1 201 Created"
[2026-03-07 09:00:46] INFO: 09:00:46 - INFO - HTTP Request: POST https://api.replicate.com/v1/models/ibm-granite/granite-3.1-8b-instruct/predictions "HTTP/1.1 201 Created"
[2026-03-07 09:00:46] INFO: 09:00:46 - INFO - HTTP Request: POST https://api.replicate.com/v1/models/ibm-granite/granite-3.1-8b-instruct/predictions "HTTP/1.1 201 Created"
[2026-03-07 09:00:47] INFO: 09:00:47 - INFO - HTTP Request: GET https://stream-b.svc.ric2.c.replicate.net/v1/streams/6pqjtlorebckojgbhvyzcdi

In [ ]:
What is adaptive leadership

: 

: 

: 

: 

: 

...

: 

: 

: 

: 

: 